# DTSC 520 · Module 4

# pandas

Modules 2 and 3 gave you ten students. You held them in dictionaries, then in
NumPy arrays, and everything worked, because ten students chosen for a lesson
are tidy by construction.

This module opens the actual extract. **One hundred and two rows**, the same
students, the same columns - and every problem that ten clean records were
hiding.

By the end you will have taken that file from "unusable" to "ready to analyze",
and you will be able to say exactly what you changed and why. That record is what lets somebody else tell your finding from a guess.

## How to use this notebook

Same conventions as Modules 2 and 3.

**Predict cells.** Commit to an answer before you run the next cell. In this
module the prediction is usually about a row count, and the gap between what you
expected and what you got is the lesson.

**`%%expect` cells.** Some cells are supposed to fail, and say so on their first
line. pandas has a small number of errors you will meet constantly - a missing
label, an ambiguous truth value, arithmetic on text - and it is worth meeting
them on purpose here rather than at midnight.

**Takeaway cells.** The things worth remembering after the syntax fades.

One habit specific to this module: **print the shape before and after anything
that filters, drops or merges.** Most of what goes wrong in pandas does not
raise an error. It just returns fewer rows than you meant, and says nothing.

Run the setup cell below first.

## What you already know that transfers

You are not starting from zero here. Three things carry straight over:

| From | Becomes |
|---|---|
| Module 2's `.get(key, 0) + 1` counting | `value_counts()` and `groupby` |
| Module 2's `is None` guard | `isna()` and the missing-data decision |
| Module 3's boolean masks | the same masks, now with labels attached |
| Module 3's views versus copies | `SettingWithCopyWarning` |

pandas is not a new set of ideas. It is the ideas you have, with an index
bolted on and the loops written for you.

In [ ]:
# DTSC 520 setup. Run this once, first, every session. Sets up %%expect.
import builtins, sys, traceback
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

@register_cell_magic
def expect(name, cell):
    """%%expect ErrorName -- run a cell that is supposed to fail."""
    want, rule = name.strip() or "Exception", "─" * 64
    try:
        code = compile(cell, "<expected to fail>", "exec")
    except SyntaxError as e:
        raised, tb = e, None
    else:
        try:
            exec(code, get_ipython().user_ns)
            print(f"{rule}\n  NO ERROR RAISED. We expected a {want} here.\n{rule}")
            return
        except BaseException as e:
            raised, tb = e, e.__traceback__.tb_next
    print(f"{rule}\n  EXPECTED ERROR. This cell is supposed to fail.\n{rule}")
    traceback.print_exception(type(raised), raised, tb, limit=4, file=sys.stdout)
    if not isinstance(raised, getattr(builtins, want, ())):
        print(f"\n  (note: expected {want}, got {type(raised).__name__})")

print("Setup complete. %%expect is ready.")

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 12)      # keep output readable in a notebook
pd.set_option("display.max_columns", None)  # never hide a column - the extract has 15
pd.set_option("display.width", 100)

print(f"pandas {pd.__version__}")
print(f"numpy  {np.__version__}")

## Sources and acknowledgments

Some of this module is taken closely from published sources, some is adapted,
and some is original to Eastern.

- **Wes McKinney, *Python for Data Analysis*, 3rd edition.** The structure of
  the selection and cleaning sections follows McKinney's treatment. McKinney
  wrote pandas, and his ordering of `.loc` before `.iloc` is deliberate and
  better than the alternative.
- **Jake VanderPlas, *Python Data Science Handbook*.** The missing-data,
  hierarchical-indexing, concatenation and pivot-table material is adapted from
  it. Text is CC BY-NC-ND, so it is paraphrased rather than copied; the code is
  MIT.
- **Allen Downey, *Think Python* 3e** (CC BY-NC-SA 4.0), for the error-reading
  approach carried through from Module 2.
- **Original to Eastern:** the cohort narrative, the diagnosis-before-cleaning
  sequence in sections 4 to 8, every exercise, and the cleaning decisions the
  capstone then holds you to.

## A word about AI assistants

You will use them. Let's be direct about how.

First, be sure to review Eastern's LLM policies. These will be on syllabi and
in the program handbook.  *It is your responsibility to know these policies.
If you violate them you could face academic discipline including expulsion*

**Good uses.** Paste an error message into an llm and ask what it means. Ask for a second
explanation of a concept when mine didn't land. Ask *why* code works, not just
whether it does. Ask it to generate extra practice problems.

**Bad use.** Pasting an exercise prompt in and submitting what comes back. You
will pass the exercise and fail the course - not because you got caught, but
because every module here assumes you can already do the one before it, and the
capstone assumes all of them.

The line is this: **you are accountable for being able to explain every line you
submit.** If you can't walk me through it, it isn't yours yet. When a model
writes something you don't understand, that's a signal to ask a follow-up
question, not to move on.

More than anything, AI should be used to **learn** - if you're using it without
learning, you're doing it wrong and will be hurt in the long run.

Where I think an assistant is genuinely useful, I'll say so in the exercise.

**How this notebook was made.** The teaching material here is
drawn from the instructor's own course materials and from the textbooks
acknowledged below. AI systems (Claude and Gemini) were used to compare those
sources for concept coverage and divergence, and to draft a notebook
synthesizing them. The sequence, the pedagogy, and every decision about what to
teach are the instructor's. The result has been reviewed and corrected by
Eastern faculty and staff, and verified to execute end to end.

## The vocabulary

pandas has a small vocabulary and uses it
precisely. Its error messages assume you know these, and so does every answer
you will find online.

- **Series** - A one-dimensional array with an index. One column of a table, with labels attached to the values.
- **DataFrame** - A set of Series sharing one index. The table, and the object most pandas work is about.
- **index** - The labels attached to a DataFrame's rows. Identifiers, not positions - which is why `.loc` and `.iloc` differ.
- **.loc** - Selection by **label**. `df.loc[1003]` asks for the row whose index is 1003, and includes the endpoint when slicing.
- **.iloc** - Selection by integer **position**. `df.iloc[0]` asks for the first row, whatever its label.
- **dtype** - The type of a column. `object` almost always means text got into something you expected to be numeric.
- **NaN** - Not a Number: pandas' marker for a missing value. It is a float, and it propagates through arithmetic.
- **missing at random** - When whether a value is missing has nothing to do with the outcome. If it is not true, dropping rows biases the answer.
- **boolean mask** - A Series of `True`/`False` used to filter rows. Combine with `&` and `|`, never `and` and `or`.
- **groupby** - Split rows into groups, apply a summary to each, combine the results. The dictionary-counting pattern, industrialised.
- **aggregation** - The summary applied to each group - `mean`, `count`, `median`. Named aggregation gives the output readable column names.
- **merge** - Joining two tables on a shared key. Check the key is unique first: a repeated key multiplies rows.
- **inner join** - Keeps only rows present in both tables. Rows without a match are dropped silently, so check the row count.
- **left join** - Keeps every row of the left table, filling gaps with `NaN`. Makes a missing match visible instead of absent.
- **pivot table** - A groupby arranged as a grid: one variable down the side, another across the top, a statistic in the cells.
- **chained assignment** - Writing `df[mask]["col"] = x`. It edits a temporary copy and silently does nothing. Use `df.loc[mask, "col"] = x`.
- **target leakage** - Letting information about the outcome into a predictor. The model looks excellent in testing and fails in use.

Module 3 ended with an engagement index: three arrays,
combined, ranked. It worked. But look at what you had to hold in your head to
make it work - that `logins[3]`, `hours[3]`, and `ug_gpa[3]` all referred to the
same student, and that position 3 meant Amara Okonkwo because a separate `names`
array said so.

Nothing enforced that. If you had sorted one array and not the others, every
number would still have been a number, every operation would still have run, and
every answer would have been wrong. Silently.

That is the problem pandas exists to solve. It attaches labels to data and then
refuses to let them come apart.

---
# 1 · Why pandas exists

Here are two of the Module 3 arrays and the sorting mistake described above.

## The alignment problem, demonstrated

In [ ]:
names  = np.array(["Amara Okonkwo", "Devin Castellanos", "Priya Raghunathan"])
ug_gpa = np.array([3.02, 1.99, 2.80])
logins = np.array([3.4, 0.4, 4.4])

# Rank students by GPA - but only sort two of the three arrays
order  = np.argsort(-ug_gpa)
print(names[order])
print(ug_gpa[order])
print(logins)          # <- never reordered

> **Predict.** Is `logins` now still lined up with `names`? What would you compute if you used both?
>
> Commit to an answer before you run the next cell.

In [ ]:
for i in range(3):
    print(f"{names[order][i]:20} gpa {ug_gpa[order][i]:.2f}   logins {logins[i]}")

**What happened.** Priya's GPA is now printed next to Amara's login count.

Nothing failed. No exception, no warning, no obviously wrong number - Priya
plausibly could log in 3.4 times a week. This is the most dangerous class of bug
in data work: the kind that produces a publishable-looking answer.

The fix is not "be careful." The fix is a data structure that carries the label
with the value.

## The same thing in pandas

In [ ]:
df = pd.DataFrame({
    "name":   ["Amara Okonkwo", "Devin Castellanos", "Priya Raghunathan"],
    "ug_gpa": [3.02, 1.99, 2.80],
    "logins": [3.4, 0.4, 4.4],
})

df.sort_values("ug_gpa", ascending=False)

**Takeaway.** **One sort, every column moves together.** The row is the unit,
not the array.

That is the whole argument for pandas in one line. Everything else - the
selection syntax, the grouping, the merging - is built on the fact that a label
and its values cannot be separated by accident.

There is a cost, and it is worth naming now rather than
letting you discover it as confusion later. NumPy has one way to index. pandas
has three, they look almost identical, and two of them will do something you did
not intend at least once each. Section 3 is about that, and it is the section
worth slowing down in.

---
# 2 · Series and DataFrame

pandas has exactly two objects you need. Everything else is a method on one of
them.

- A **Series** is a one-dimensional array with an index. Think of one column.
- A **DataFrame** is a set of Series sharing one index. Think of the table.

Start with the Series.

In [ ]:
gpa = pd.Series([3.02, 1.99, 2.80],
                index=["Amara Okonkwo", "Devin Castellanos", "Priya Raghunathan"])
gpa

Two things to notice in that output: the values on the right, and
the **index** on the left. The index is not decoration and it is not a row
number - it is a label you can select with.

In [ ]:
print(gpa["Devin Castellanos"])
print(gpa.values)         # the underlying NumPy array - Module 3 is still in there
print(gpa.index)

**Takeaway.** `.values` gives you back the NumPy array. A Series *is* a NumPy
array plus an index, and every vectorized operation from Module 3 still
works - `gpa * 2`, `gpa[gpa > 3]`, `gpa.mean()`. You have not lost anything.

> **Predict.** `gpa` has three entries. What does `gpa[0]` give you - and is that the same as `gpa["Amara Okonkwo"]`?
>
> Commit to an answer before you run the next cell.

In [ ]:
print(gpa.iloc[0])                 # by position
print(gpa["Amara Okonkwo"])        # by label

**What happened.** Both give 3.02, because Amara is the first entry.

They agree here, which is exactly why this is worth flagging. When the index is
made of strings, position and label are different questions that happen to have
the same answer. When the index is made of *numbers* - which it will be, in
about ten minutes, when we load a file keyed on `student_id` - they come apart,
and `gpa[1001]` becomes genuinely ambiguous.

That ambiguity is why `.loc` and `.iloc` exist. Section 3.

## DataFrame

A DataFrame is the table. The most common way to build one by hand is from a
dictionary of columns.

In [ ]:
students = pd.DataFrame({
    "name":     ["Amara Okonkwo", "Devin Castellanos", "Priya Raghunathan"],
    "ug_gpa":   [3.02, 1.99, 2.80],
    "logins":   [3.4, 0.4, 4.4],
    "internships": [0, 0, 2],
})

students

In [ ]:
print(students.shape)        # (rows, columns) - same attribute as Module 3
print(students.columns)
print(students.index)

Select one column and you get a Series back. Select a list of
columns and you get a DataFrame.

In [ ]:
print(type(students["ug_gpa"]))
print(type(students[["ug_gpa", "logins"]]))

**Takeaway.** **Single brackets give a Series, double brackets give a
DataFrame.** `students[["ug_gpa"]]` is a one-column table; `students["ug_gpa"]`
is a column.

This trips people constantly, and the error it produces later is usually about a
missing attribute rather than about brackets.

**Your turn.**

Build a DataFrame called `cohort` with these three columns:
`name`, `hs_gpa`, `hs_clubs`, for any three students you like. Then print its
shape and select just the `hs_gpa` column as a Series.

In [ ]:
### ENTER CODE HERE ###

Selection is where pandas earns its reputation for being
fiddly, and the reputation is deserved. There are three ways to get at a piece
of a DataFrame, they are spelled almost the same, and the differences only
matter in the cases where getting it wrong is expensive.

Learn them in one sitting, deliberately, and you will not have to think about it
again.

---
# 3 · Selection: `[]`, `.loc`, `.iloc`

Give the DataFrame a meaningful index first, because that is when the
distinctions start to bite.

In [ ]:
students = pd.DataFrame({
    "name":     ["Amara Okonkwo", "Priya Raghunathan", "Devin Castellanos"],
    "ug_gpa":   [3.02, 2.80, 1.99],
    "logins":   [3.4, 4.4, 0.4],
}, index=[1003, 1024, 1089])       # student_id, exactly as the extract is keyed

students.index.name = "student_id"
students

Now the index is made of integers that are **identifiers, not
positions**. Amara is student 1003; she is also row 0. Those are different facts
and they are about to disagree.

**`.loc` selects by label.**

In [ ]:
students.loc[1003]                 # the student whose id is 1003

**`.iloc` selects by position.**

In [ ]:
students.iloc[0]                   # whatever is in the first row

> **Predict.** What do you think `students.loc[0]` does? And `students.iloc[1003]`?
>
> Commit to an answer before you run the next cell.

In [ ]:
%%expect KeyError
students.loc[0]

In [ ]:
%%expect IndexError
students.iloc[1003]

**What happened.** Both fail, and the two error types tell you which question you
asked.

`.loc[0]` is a **KeyError**: you asked for the student whose id is 0, and there
is no such student. `.iloc[1003]` is an **IndexError**: you asked for the
1,004th row of a three-row table.

Read those two words as the question you asked. KeyError means "no such label."
IndexError means "no such position."

**Takeaway.** **`.loc` is labels, `.iloc` is integer position.** The `i` is
for integer, and that is the whole mnemonic.

Prefer `.loc`. Position is an accident of how the file happened to be sorted;
a label is a fact about the student. Code written with `.loc` keeps meaning the
same thing after a `sort_values`, and code written with `.iloc` does not.

## Two axes

Both take a row selector and a column selector, separated by a comma.

In [ ]:
print(students.loc[1003, "ug_gpa"])            # one student, one column
print()
print(students.loc[[1003, 1024], ["name", "ug_gpa"]])   # some rows, some columns

Slicing with `.loc` is **inclusive of the endpoint**, which is not
how Python slicing works anywhere else. This is a real inconsistency in the
library, not a misunderstanding on your part.

In [ ]:
print(students.loc[1003:1024])     # includes 1024
print()
print(students.iloc[0:2])          # excludes position 2, like normal Python

**Takeaway.** `.loc[a:b]` **includes** `b`. `.iloc[a:b]` excludes it, like
every other slice in Python.

The reason is that with labels there is no "one past the end" to point at. It is
still a trap, and it is worth checking the row count after any `.loc` slice.

### One caution about label slices

That worked cleanly because the index is **sorted**. A label slice does not
sort - it finds where `a` sits, finds where `b` sits, and takes everything
between them *in the order the rows happen to be stored*.

In [ ]:
shuffled = students.loc[[1003, 1089, 1024]]     # same rows, different order
print(f"index sorted? {shuffled.index.is_monotonic_increasing}")
shuffled.loc[1003:1024]

**What happened.** Devin (1089) is inside a slice that reads `1003:1024`, even
though 1089 is larger than 1024.

Nothing is broken. The slice took every row between the *position* of 1003 and
the *position* of 1024, and on this ordering Devin sits between them. On a
sorted index that is the same thing as "every label in the range." On an
unsorted one it is not.

Sort the index before slicing it, with `.sort_index()`, or select an explicit
list of labels instead.

## Plain `[]`

Plain brackets are the ambiguous one. On a DataFrame they select **columns** -
except when given a slice or a boolean mask, when they select **rows**.

In [ ]:
print(students["ug_gpa"].head(2))      # a column
print()
print(students[students.ug_gpa > 2.5])  # rows, via a boolean mask

**Takeaway.** Use `[]` for two things only: selecting a column by name, and
filtering with a boolean mask. For anything else, say `.loc` or `.iloc` and be
explicit. Ambiguity in selection code is how you get a bug that only shows up
after someone sorts the data.

**Your turn.**

Using `students`, and using `.loc` rather than `[]`:

1. Select Priya Raghunathan's row by her student_id, 1024.
2. Select the `name` and `logins` columns for students 1003 and 1024.
3. Select every student with more than 1.0 logins per week.

In [ ]:
### ENTER CODE HERE ###

Everything so far has used three students typed out by
hand - clean, small, and completely unlike anything you will be given.

The next section opens the real file.

---
# 4 · Loading the extract

Institutional Research has sent the full file. It is the same cohort you have
been working with since Module 2 - the same students, the same columns, the same
`student_id`s - but all of it, exactly as it came out of the system.

`pd.read_csv` reads a comma-separated file into a DataFrame.

In [ ]:
raw = pd.read_csv("data/cohort_extract.csv")
raw.head()

If that raised a `FileNotFoundError`, the notebook is not running
from the `notebooks/` folder. Reading a file is the first place a notebook's
working directory ever matters, so find out where you actually are:

In [ ]:
import os
print(os.getcwd())

The path in `read_csv` is interpreted **relative to that
directory**. Either start the notebook from `notebooks/`, or pass an absolute
path. This is the single most common thing that stops a working notebook from
working on somebody else's machine.

## The index changed under you

Section 3 built a DataFrame whose index *was* the `student_id`, so
`students.loc[1003]` meant "the student with id 1003." Check what `read_csv`
gave you instead.

In [ ]:
print(raw.index)
print()
print(raw.columns.tolist()[:4])

> **Predict.** `student_id` came in as a column, and the index is a plain `RangeIndex`. So what does `raw.loc[1003]` do now - and what about `raw.loc[7]`?
>
> Commit to an answer before you run the next cell.

In [ ]:
%%expect KeyError
raw.loc[1003]

In [ ]:
row = raw.loc[7]
print(row[["student_id", "name"]])

**What happened.** The first fails, which is survivable. **The second is the
dangerous one.**

`raw.loc[1003]` raises a KeyError because there is no row *labeled* 1003 - the
labels are now 0 to 101. Annoying, but it tells you immediately.

`raw.loc[7]` succeeds, and returns student 1008. You asked for a label, the
label 7 exists because the index is just row numbers, and you got a student who
has nothing to do with the number you typed. No error, no warning, wrong
student.

The lesson from section 3 has not changed - `.loc` is still labels. What changed
is *what the labels are*.

### Two honest ways to fix it

**Either** filter on the column, which works on any index:

In [ ]:
raw[raw.student_id == 1003][["student_id", "name", "ug_gpa"]]

**Or** make `student_id` the index, so `.loc` means what you want:

In [ ]:
indexed = raw.set_index("student_id")
indexed.loc[1003, ["name", "ug_gpa"]]

`read_csv` can do that in one step, which is usually better because
it never lets the confusing state exist:

```python
raw = pd.read_csv("data/cohort_extract.csv", index_col="student_id")
```

We keep `student_id` as a column for now, deliberately - section 5 has to check
whether it is even usable as a key, and you cannot easily check an index for
duplicates you have already told pandas to treat as the index.

**Takeaway.** **Before you use `.loc`, know what the index is.** `df.index`
takes one second and answers it.

A DataFrame read from a file almost always arrives with a `RangeIndex`, which
means integer labels that look exactly like positions. That is the one situation
where `.loc` and `.iloc` agree on every input and neither one warns you that you
asked the wrong question.

## First look, before anything else

Four commands, always, in this order, before you compute a single statistic.

In [ ]:
raw.shape

> **Predict.** You worked with 10 students in Modules 2 and 3. The registrar says this cohort has 100 students. What number do you expect `.shape` to report?
>
> Commit to an answer before you run the next cell.

In [ ]:
print(f"rows    {raw.shape[0]}")
print(f"columns {raw.shape[1]}")
print(f"distinct student_ids {raw.student_id.nunique()}")

`.nunique()` returns the **count** of distinct values.
`.unique()` returns the distinct values themselves, as an array - useful when
the column is categorical and you want to see the categories rather than count
them.

In [ ]:
print(raw.success.unique())        # the values
print(raw.success.nunique())       # how many there are

**What happened.** **102 rows, but only 100 distinct students.**

The file has more rows than there are people in it. That is not a rounding
error and it is not something to shrug at - it means at least one student is
counted twice in every average you compute from this file.

Do not fix it yet. Write it down. We are going to catalogue every problem in
this file before changing any of it, and section 6 is where the fixing
happens.

**Takeaway.** **Diagnose the whole file before you clean any of it.**

The temptation is to fix each problem the moment you find it. Resist it. Cleaning
changes what the next diagnostic reports, so a half-cleaned file tells you a
half-true story about itself - and you will not be able to reconstruct what the
raw data actually looked like.

Catalogue first. Clean once, deliberately, in one place. That is also what makes
your cleaning reproducible by somebody else, which is the actual standard.

## `.info()` - the single most useful command in pandas

In [ ]:
raw.info()

Read that output as three questions:

- **How many non-null values does each column have?** Any column short of the
  full row count has missing data.
- **What is each column's `Dtype`?** `object` on a column you expected to be
  numeric means something non-numeric got in - the same lesson as Module 3's
  `<U21` arrays, in a new costume.
- **How much memory?** Rarely interesting at this size. It matters at a million
  rows.

Two columns in that output should bother you. Find them before reading on.

In [ ]:
raw.dtypes

> **Predict.** `submission_rate` holds values like 84%. What dtype did pandas give it, and what happens if you try to take its mean?
>
> Commit to an answer before you run the next cell.

In [ ]:
%%expect TypeError
raw.submission_rate.mean()

**What happened.** `submission_rate` is `object` - pandas's label for "these are
Python objects, most likely strings."

The percent sign makes each value text. `"84%"` is not a number, so the whole
column is not numbers, so arithmetic on it fails. One character per cell has
disabled an entire column.

This is the single most common real-world data problem there is, and it is why
`.info()` is the first thing you run. Section 7 fixes it.

**Your turn.**

Answer these from `raw`, using code rather than by reading the
`.info()` output:

1. How many columns have at least one missing value?
2. Which column has the most missing values, and how many?
3. What percentage of rows have a missing `logins_per_week`?

In [ ]:
### ENTER CODE HERE ###

A cleaning script that someone else cannot check is not
much better than no cleaning at all. So this section produces a written
inventory of everything wrong with the file, with the code that found each
problem next to it.

That inventory is a deliverable. The capstone asks for it directly, and it is
the thing a colleague reads when they want to know whether to trust your
numbers.

---
# 5 · Diagnosis

Four problems, four diagnostics. Run each one and record what it says.

## 1 · Duplicate rows

`.duplicated()` returns a boolean Series marking every row that has appeared
before. It is a mask, exactly like Module 3's.

In [ ]:
dupes = raw.duplicated()

print(f"duplicate rows: {dupes.sum()}")
raw[dupes]

`.duplicated()` marks the *second and later* copies, not the first,
which is why two duplicated students give two marked rows rather than four. To
see every copy together, pass `keep=False`.

In [ ]:
all_copies = raw[raw.duplicated(keep=False)].sort_values("student_id")
all_copies[["student_id", "name", "ug_gpa", "course_score"]]

**Takeaway.** **`keep="first"` (the default) marks the extras. `keep=False`
marks every member of a duplicated group.**

Use the default to *remove* duplicates. Use `keep=False` to *inspect* them,
because you usually want to see what the duplicates disagree about - and here,
they agree about everything, which tells you this is a copy-paste export glitch
rather than two different students who share an id.

## 2 · Missing values

In [ ]:
missing = raw.isna().sum()
missing[missing > 0]

> **Predict.** Three columns are missing the same number of values. What does that suggest about how the data went missing?
>
> Commit to an answer before you run the next cell.

In [ ]:
lms = ["logins_per_week", "time_on_task_hrs", "submission_rate"]

print(raw[lms].isna().sum())
print()
print(f"rows missing ALL three:  {raw[lms].isna().all(axis=1).sum()}")
print(f"rows missing ANY of them: {raw[lms].isna().any(axis=1).sum()}")

**What happened.** The same rows are missing all three, so this is not three
separate problems - it is one problem. An entire LMS record failed to attach for
some students.

`.all(axis=1)` and `.any(axis=1)` are Module 3's `any` and `all`, with `axis=1`
meaning "across the columns, per row." The idea has not changed.

### The dangerous part

Missing data is only harmless when it is missing *at random*. Check whether it
is.

In [ ]:
has_lms = raw.logins_per_week.notna()

print("mean course_score:")
print(f"  students WITH an LMS record:    {raw.loc[has_lms, 'course_score'].mean():.1f}")
print(f"  students WITHOUT an LMS record: {raw.loc[~has_lms, 'course_score'].mean():.1f}")
print()
print("success rate:")
print(f"  with:    {raw.loc[has_lms, 'success'].mean():.2f}")
print(f"  without: {raw.loc[~has_lms, 'success'].mean():.2f}")

### What `dropna()` would actually cost

That gap is abstract until you watch it move a number you care about.

In [ ]:
naive = raw.dropna()

print(f"rows before dropna(): {len(raw)}")
print(f"rows after  dropna(): {len(naive)}")
print()
print(f"mean course_score, all students:   {raw.course_score.mean():.2f}")
print(f"mean course_score, after dropna(): {naive.course_score.mean():.2f}")
print()
print(f"success rate, all students:   {raw.success.mean():.2f}")
print(f"success rate, after dropna(): {naive.success.mean():.2f}")

**What happened.** One line of "tidying up" made the cohort look better than it
is.

Nobody edited a score. Every number in `naive` is a real number about a real
student. The cohort simply lost a group that was doing worse, and the average of
what remained went up on its own.

If you reported that second figure, you would be telling the Cabinet that
students are succeeding at a rate the data does not support - and you would have
no idea you had done it, because `dropna()` is not a suspicious-looking
function.

**Takeaway.** **The students with no engagement record are not a random
sample.** They score differently and they succeed at a different rate.

So `dropna()` here would not just make the dataset smaller - it would make it
*wrong*, by quietly deleting a group that differs on the outcome you are trying
to explain. Any conclusion you drew about engagement afterwards would be
measuring, in part, who happened to have a record at all.

This is the most important idea in the module. Section 8 is about what to do
instead, and there is no answer that costs nothing.

## 3 · Wrong dtype

In [ ]:
print(raw.submission_rate.dtype)
print(raw.submission_rate.head())
print()
print(f"distinct non-null values: {raw.submission_rate.nunique()}")

## 4 · Impossible values

A dtype can be right and the values still be nonsense. `.describe()` gives the
five-number summary of every numeric column, and you read it looking for
impossibilities.

In [ ]:
raw[["hs_gpa", "hs_attendance", "ug_gpa", "course_score"]].describe()

> **Predict.** `hs_attendance` is a rate - the proportion of days a student attended. What is the largest value it could legitimately take, and what does the table say the maximum actually is?
>
> Commit to an answer before you run the next cell.

In [ ]:
bad = raw[raw.hs_attendance > 1.0]

print(f"rows with attendance above 100%: {len(bad)}")
bad[["student_id", "name", "hs_attendance"]]

**What happened.** An attendance rate of 1.4 means attending 140% of school days.

Nothing in the file is going to flag this. The dtype is `float64`, which is
correct; the value is a number, which is correct; every computation you run on
it will succeed. Only knowing what the column *means* catches it.

That is why `.describe()` belongs in the diagnosis step. You are not looking at
the mean - you are looking at the min and max and asking whether they are
possible.

## The inventory

Four problems, and each needs a decision rather than a reflex.

In [ ]:
inventory = pd.DataFrame([
    {"problem": "duplicate rows",      "n": int(raw.duplicated().sum()),
     "column": "(whole row)",          "decision": "drop the extra copies"},
    {"problem": "missing LMS record",  "n": int(raw.logins_per_week.isna().sum()),
     "column": "logins/hours/submission", "decision": "NOT dropna - see section 8"},
    {"problem": "stored as text",      "n": int(raw.submission_rate.notna().sum()),
     "column": "submission_rate",      "decision": "strip % and convert"},
    {"problem": "impossible value",    "n": int((raw.hs_attendance > 1.0).sum()),
     "column": "hs_attendance",        "decision": "treat as missing"},
])

inventory

**Your turn.**

One diagnostic is missing from the list above: are the
`student_id` values you would use as a key actually usable as one?

Check whether `student_id` is unique in `raw`, and if it is not, show which ids
repeat. Then say in a sentence why this matters before any merge.

In [ ]:
### ENTER CODE HERE ###

You now have a written account of what is wrong with this
file, and not one value has been changed. That order matters: the inventory is
what lets you explain your cleaning later, and it is what a reviewer checks your
work against.

The next three sections fix these four problems, one decision at a time.

The inventory is written, so the arguing is over. What
follows is bookkeeping: four problems, four decisions, each one applied once and
recorded.

Two rules govern all of it, and they exist because notebooks get run out of
order. The raw data is never overwritten, and nothing is modified in place. Every
cleaning step takes a DataFrame and returns a new one with a new name. That way
any cell can be re-run at any time and still mean what it says.

---
# 6 · Cleaning 1: structure

Start with the two problems that are about the shape of the table rather than
the values in it: duplicate rows, and an impossible number.

## Why `raw` keeps its name

Section 5 measured `raw`. If cleaning reassigns `raw`, then scrolling up and
re-running the diagnosis reports something different, and you lose the ability to
say what the file looked like when it arrived.

So every step below produces a **new name**. This costs nothing and it is what
makes the cleaning reproducible.

In [ ]:
print(f"raw is still untouched: {raw.shape[0]} rows")

## Dropping the duplicates

`drop_duplicates()` keeps the first occurrence of each duplicated row and
discards the rest.

In [ ]:
deduped = raw.drop_duplicates()

print(f"before: {len(raw)} rows")
print(f"after:  {len(deduped)} rows")
print(f"removed: {len(raw) - len(deduped)}")

### Not `inplace=True`

`drop_duplicates` accepts `inplace=True`, and almost every tutorial you will
find uses it. Do not.

**Takeaway.** **Avoid `inplace=True` in pandas, and especially in a
notebook.**

Three reasons, in increasing order of importance:

1. It does not usually save memory. pandas generally builds the new object
   anyway and then rebinds the name, so the promised efficiency mostly is not
   real.
2. It returns `None`, which breaks method chaining. `df.drop_duplicates(inplace=True).head()`
   is an `AttributeError`, and the message will not mention `inplace`.
3. **It changes an object you cannot see.** In a notebook, cells run in whatever
   order you click them. An in-place operation run twice, or run before a cell
   you then scroll up and re-execute, leaves you holding a DataFrame that does
   not match the code on screen. That is the single most common way a notebook
   starts producing numbers nobody can reproduce.

Explicit reassignment is one extra word and the state is always legible.

## The index after dropping rows

Look at what the row labels do.

In [ ]:
print(deduped.index[-6:].tolist())
print(f"rows: {len(deduped)}, last label: {deduped.index[-1]}")

> **Predict.** The table now has 100 rows. What is the largest row label, and why is it not 99?
>
> Commit to an answer before you run the next cell.

In [ ]:
gaps = [i for i in range(len(raw)) if i not in set(deduped.index)]
print(f"labels missing from the index: {gaps}")

**What happened.** Dropping rows removes their labels but does not renumber
anything. The index still runs to 101 with two holes in it, because labels are
identifiers, not positions - the same distinction from section 4.

Usually harmless. It bites when you later assume `.iloc[i]` and `.loc[i]` agree,
or when you write the file out and someone reads the index as a row count.

In [ ]:
clean = deduped.reset_index(drop=True)

print(f"rows: {len(clean)}, last label: {clean.index[-1]}")

**Takeaway.** **`reset_index(drop=True)` renumbers the rows 0 to N-1.**

Without `drop=True` the old index is kept as a new column called `index`, which
is occasionally what you want and usually clutter.

Worth knowing what you are giving up: those original labels were the only link
back to line numbers in the source file. If you need to tell somebody which row
of their CSV was malformed, capture it before you reset.

### The index plan for this module

Three stages, decided once so it does not have to be revisited:

| Stage | Index | Why |
|---|---|---|
| on load | `RangeIndex` | duplicates are visible as rows, and `student_id` can be checked for uniqueness like any other column |
| after cleaning | `RangeIndex`, reset | contiguous, no holes |
| before merging | `student_id` | a join needs a real key, and by then it is known to be unique |

We are at stage two. Stage three arrives in section 11.

## The impossible value

One student has an `hs_attendance` of 1.4. Here is the wrong way to fix it, shown
on purpose.

In [ ]:
attempt = clean.copy()
attempt[attempt.hs_attendance > 1.0]["hs_attendance"] = np.nan

print(f"rows still above 1.0: {(attempt.hs_attendance > 1.0).sum()}")

> **Predict.** That cell probably printed a warning. Ignore the warning for a moment and read the number underneath it. Did the edit work?
>
> Commit to an answer before you run the next cell.

**What happened.** **The edit did nothing.** There is still one row above 1.0.

`attempt[attempt.hs_attendance > 1.0]` builds a **new** DataFrame containing the
matching rows. Assigning into that new object writes to the temporary copy, which
is then discarded. The original is untouched.

This is Module 3's views-versus-copies problem wearing different clothes. There
you learned that a NumPy slice may be a view of the original; here the danger is
the reverse, a thing that looks like part of the original but is a copy.

pandas does try to warn you. **What it says depends on your version** - older
pandas raises `SettingWithCopyWarning`, and newer pandas with Copy-on-Write
raises `ChainedAssignmentError` instead. Do not memorise either name. Memorise
the symptom: *you assigned, and nothing changed.*

The tell in the code is the **two sets of brackets in a row**. That is chained
indexing, and it is never what you want on the left of an `=`.

### The fix: one `.loc`, both axes at once

In [ ]:
clean = clean.copy()
clean.loc[clean.hs_attendance > 1.0, "hs_attendance"] = np.nan

print(f"rows above 1.0: {(clean.hs_attendance > 1.0).sum()}")
print(f"hs_attendance now missing for: {clean.hs_attendance.isna().sum()} student(s)")
print(f"max attendance: {clean.hs_attendance.max()}")

**Takeaway.** **Select rows and columns in a single `.loc`, never with two
bracket pairs.**

```python
df.loc[mask, "col"] = value      # one operation, writes to df
df[mask]["col"] = value          # two operations, writes to a temporary
```

The rule is easy to apply: if you see `][` on the left of an `=`, it is wrong.

Notice the decision made above: the impossible value became `NaN`
rather than being deleted or corrected.

We cannot know what the attendance rate should have been, so inventing one would
be fabrication. Deleting the student would throw away their other twelve valid
fields. Marking the single bad value as missing keeps everything we do know and
records honestly what we do not.

**Your turn.**

`hs_gpa` and `ug_gpa` are both on a 4.0 scale, so any value above 4.0 is
impossible in the same way.

1. Check whether `clean` contains any.
2. Write the `.loc` statement that would set them to `NaN` if it did.
3. Confirm afterwards that the count is zero either way.

In [ ]:
### ENTER CODE HERE ###

Two problems down. Both were structural, and both had a
defensible answer that did not require a judgment call about the students
themselves.

The next one is different. It is a column that is not broken at all - it simply
arrived in the wrong shape.

---
# 7 · Cleaning 2: the column that is text

`submission_rate` holds values like `"84%"`. Every one is meaningful, none is
missing in the ordinary sense, and the whole column is unusable because of one
character per cell.

In [ ]:
print(clean.submission_rate.dtype)
print(clean.submission_rate.head(4).tolist())

## Why one character disables a column

pandas gives a column a single dtype. One string forces the whole column to
`object`, and `object` means "arbitrary Python objects", which supports almost
no arithmetic.

In [ ]:
%%expect TypeError
clean.submission_rate.mean()

**Takeaway.** **A `%` or a `$` or a comma in a number column silently makes
it text.**

Nothing is malformed here. A human reading `84%` sees a number. pandas sees a
five-character string, and every mathematical method on the column is disabled
from the moment the file is read.

This is why `.info()` is the first thing you run. The dtype tells you before you
waste an hour on a calculation that cannot work.

## The `.str` accessor

A Series of strings exposes Python's string methods through `.str`, applied to
every element at once. This is Module 3's vectorization, applied to text.

In [ ]:
print(clean.submission_rate.str.rstrip("%").head(4).tolist())

> **Predict.** Those still print with quote marks. Is the column numeric yet?
>
> Commit to an answer before you run the next cell.

In [ ]:
stripped = clean.submission_rate.str.rstrip("%")
print(stripped.dtype)

**What happened.** Still `object`. Removing the `%` left strings that happen to
look like numbers, which is not the same as numbers.

`"84"` and `84` are different objects. The conversion is a second, separate
step - and this is the step everybody forgets.

## `.astype()` converts

In [ ]:
rate = clean.submission_rate.str.rstrip("%").astype(float) / 100

print(rate.dtype)
print(rate.head(4).tolist())
print(f"mean submission rate: {rate.mean():.3f}")

Assign it back with `.loc`-free column assignment, which is
unambiguous because it names a whole column rather than a subset of rows.

In [ ]:
clean = clean.copy()
clean["submission_rate"] = clean.submission_rate.str.rstrip("%").astype(float) / 100

clean[["student_id", "name", "submission_rate"]].head(4)

## Why that did not fail on the missing values

Seventeen students have no submission rate at all. `.astype(float)` on a column
containing blanks would normally be a problem.

In [ ]:
print(f"missing submission rates: {clean.submission_rate.isna().sum()}")

**What happened.** `read_csv` had already turned the empty cells into `NaN`, and
`NaN` is a **float**, so it passes through `.astype(float)` untouched.

The `.str` methods do the same thing - they return `NaN` for missing entries
rather than raising. That is deliberate design in pandas, and it is why the
whole chain works in one line.

It also means the missing values are still missing. Nothing here fixed them, and
section 8 has to decide what to do about that.

## When `.str` is not enough: `.apply()`

`.str.rstrip` worked because every value had the same shape. Real columns are
often messier, and then you need an arbitrary function applied per element.

In [ ]:
messy = pd.Series(["84%", "0.91", "77 %", "", "n/a", "1.0"])

def to_rate(v):
    """Convert one submission-rate value to a float in 0-1, or NaN."""
    if not isinstance(v, str) or v.strip().lower() in {"", "n/a", "na", "-"}:
        return np.nan
    v = v.strip().rstrip("%").strip()
    try:
        x = float(v)
    except ValueError:
        return np.nan
    return x / 100 if x > 1 else x

messy.apply(to_rate)

**Takeaway.** **`.apply()` runs a Python function on every element.** It is
the escape hatch: anything you can express as a function, you can apply.

It is also **slower than a vectorized method**, because it is a Python loop
wearing a method's clothing - the Module 3 lesson again. Reach for `.str`,
`.astype` and arithmetic first, and use `.apply` when the logic genuinely cannot
be expressed that way.

`.map()` is the Series equivalent and also takes a dictionary, which makes it the
natural tool for recoding categories.

**Your turn.**

`need_based_aid` is stored as `0` and `1`. For a report, those want to be
readable labels.

1. Use `.map()` with a dictionary to make a new Series of `"yes"` / `"no"`.
2. Count how many students are in each category with `value_counts()`.
3. Say in a comment why you would *not* overwrite the original column with the
   labels.

In [ ]:
### ENTER CODE HERE ###

The column is numeric, the structure is sound, and the
impossible value is gone. Three of the four problems in the inventory are
closed, and none of them required a judgment about the students.

The fourth one does, and there is no answer that costs nothing.

Section 5 established that the students without an LMS
record are not a random sample of the cohort - they score lower and they succeed
less often. That means the missing values carry information, and every way of
handling them trades one kind of wrongness for another.

This section is about choosing deliberately and writing down what you chose. It
is the part of the module the capstone actually holds you to.

---
# 8 · Cleaning 3: the missing values

Start by restating the problem against the cleaned table.

In [ ]:
lms = ["logins_per_week", "time_on_task_hrs", "submission_rate"]

missing_mask = clean[lms].isna().all(axis=1)
print(f"students with no LMS record: {missing_mask.sum()} of {len(clean)}")
print()
print(f"success rate, has record:  {clean.loc[~missing_mask, 'success'].mean():.2f}")
print(f"success rate, no record:   {clean.loc[missing_mask, 'success'].mean():.2f}")

## The three options, and what each one costs

### Option 1: drop the rows

In [ ]:
dropped = clean.dropna(subset=lms)

print(f"rows: {len(clean)} -> {len(dropped)}")
print(f"success rate: {clean.success.mean():.3f} -> {dropped.success.mean():.3f}")

### Option 2: fill with a central value

In [ ]:
filled = clean.copy()
for col in lms:
    filled.loc[:, col] = filled[col].fillna(filled[col].median())

print(f"rows kept: {len(filled)}")
print(f"success rate: {filled.success.mean():.3f}")
print(f"logins std: {clean.logins_per_week.std():.3f} -> {filled.logins_per_week.std():.3f}")

> **Predict.** Filling kept every row, so the success rate is unchanged. But one number above moved a lot. Which, and why does it matter?
>
> Commit to an answer before you run the next cell.

**What happened.** The **standard deviation fell**.

Seventeen students were just assigned exactly the median. That is seventeen
points of zero variation added to the column, so the spread shrinks and every
student who was actually missing now looks perfectly typical.

Filling does not add information. It manufactures the appearance of information,
and it does so most for the students you know least about. Any correlation
computed afterwards is diluted toward zero, which will make engagement look like
a *weaker* predictor than it is.

### Option 3: keep the rows, and record the missingness

In [ ]:
flagged = clean.copy()
flagged["lms_missing"] = flagged[lms].isna().all(axis=1)

for col in lms:
    flagged.loc[:, col] = flagged[col].fillna(flagged[col].median())

print(flagged.groupby("lms_missing")[["course_score", "success"]].mean().round(3))

**Takeaway.** **Adding a missingness flag keeps the fact that the value was
missing, which is itself a finding.**

The two groups differ on the outcome. A model given `lms_missing` can use that;
an analyst reading the table can see it. Neither is possible once the rows are
deleted, and it is invisible once the values are filled without a flag.

The cost is honesty about what the filled numbers are: they are placeholders,
not measurements, and any statement about the engagement columns has to be
qualified for that group.

## The decision

There is no option here without a cost, so the deliverable is not a clean table.
It is a clean table **plus a written record of what was traded away**.

In [ ]:
decisions = pd.DataFrame([
    {"problem": "2 duplicate rows",
     "action": "drop_duplicates(), keep first",
     "cost": "none - the rows were identical"},
    {"problem": "hs_attendance = 1.4",
     "action": "set to NaN",
     "cost": "1 student now missing an attendance value"},
    {"problem": "submission_rate stored as text",
     "action": ".str.rstrip('%').astype(float) / 100",
     "cost": "none - reversible formatting change"},
    {"problem": "17 students missing all LMS fields",
     "action": "median fill + lms_missing flag",
     "cost": "filled values are placeholders; column variance understated"},
])

decisions

**Takeaway.** **Write this table before you write a single finding.**

Every number you report later is conditional on these four choices. A reader who
disagrees with one of them needs to know it was made, and you need to be able to
say why - which is much harder to reconstruct a week later than to record now.

The capstone asks for exactly this table. Build it as you go, not afterwards from memory.

**Your turn.**

The median fill above used the median of *all* students who had a value.

1. Compute the median `logins_per_week` separately for successful and
   unsuccessful students.
2. Say, in a comment, why filling from those group medians would be a **worse**
   choice here even though it looks more careful.

In [ ]:
### ENTER CODE HERE ###

The table is clean, the decisions are recorded, and the
things that were traded away are written down where somebody else can argue with
them.

That is the whole cleaning workflow, and it is worth noticing how little of it
was about pandas syntax. Four methods did the work. The rest was deciding what
the data was allowed to become.

The table is clean. From here the work stops being about
repair and starts being about questions - and every question you can ask of a
table is some combination of three moves: pick rows, group them, and summarize
them.

This section is the first move, and you already know it. A boolean mask in
pandas is the same object it was in Module 3. The only difference is that the
rows it selects carry their labels with them.

---
# 9 · Filtering

A comparison on a column gives a Series of `True` and `False`, one per row.

In [ ]:
low_engagement = clean.logins_per_week < 1.0

print(type(low_engagement))
print(low_engagement.head())
print(f"\nmatching students: {low_engagement.sum()}")

`.sum()` on a boolean Series counts the `True` values, because
`True` is 1. That is a Module 2 fact still paying rent.

Pass the mask to the DataFrame to get the rows.

In [ ]:
clean[low_engagement][["student_id", "name", "logins_per_week", "course_score"]]

## Combining conditions

Use `&` and `|`, not `and` and `or`, and **parenthesise each condition**.

In [ ]:
at_risk = (clean.logins_per_week < 1.5) & (clean.ug_gpa < 2.5)

print(f"students low on both measures: {at_risk.sum()}")
clean[at_risk][["name", "logins_per_week", "ug_gpa", "success"]]

> **Predict.** Why `&` rather than `and`? What do you think `clean.logins_per_week < 1.5 and clean.ug_gpa < 2.5` would do?
>
> Commit to an answer before you run the next cell.

In [ ]:
%%expect ValueError
clean.logins_per_week < 1.5 and clean.ug_gpa < 2.5

**What happened.** "The truth value of a Series is ambiguous."

`and` wants to reduce each side to a single `True` or `False`, and a hundred
booleans cannot answer that question - pandas refuses to guess whether you meant
"any" or "all". `&` is the element-wise version, which is what you actually
want: compare row by row.

The parentheses matter for a duller reason. `&` binds more tightly than `<`, so
without them Python reads `1.5 & clean.ug_gpa` and fails somewhere confusing.

**Takeaway.** **`&` and `|` for masks, and parenthesise every condition.**

The same rule you learned for NumPy in Module 3. `and` and `or` are for single
`True`/`False` values; `&` and `|` are for arrays and Series of them.

## `.isin()` and `~`

`.isin()` tests membership against a list, and `~` inverts a mask.

In [ ]:
cast = ["Amara Okonkwo", "Devin Castellanos", "Priya Raghunathan"]

print(clean[clean.name.isin(cast)][["name", "ug_gpa", "logins_per_week"]])
print()
print(f"everyone else: {(~clean.name.isin(cast)).sum()} students")

## `.query()`, when the condition gets long

`.query()` takes the condition as a string. It is easier to read once you have
three or more clauses, and slower, which rarely matters at this size.

In [ ]:
clean.query("logins_per_week < 1.5 and ug_gpa < 2.5")[["name", "ug_gpa"]]

Note that inside `.query()` you write `and`, not `&` - it is a
small expression language, not Python. That is a fair reason to prefer masks
until the length of the condition makes the case for you.

**Your turn.**

Lee wants a shortlist: students who are **succeeding despite low
engagement**, since they may reveal something the engagement index misses.

1. Build a mask for `logins_per_week` below the cohort median **and**
   `success == 1`.
2. Show their names, logins, and course scores, sorted by score descending.
3. Say in a comment how many there are and whether that is few enough to be
   worth looking at individually.

In [ ]:
### ENTER CODE HERE ###

In Module 2 you counted things with a dictionary:

    counts[key] = counts.get(key, 0) + 1

That line is the whole idea of grouping. You looked at each record, decided
which bucket it belonged in, and added to that bucket's running total. It took
four lines and a loop.

`groupby` is that pattern with the loop written for you, and it does not stop at
counting - once the rows are in buckets you can take any summary you like of
each one.

---
# 10 · Grouping

Three steps, always, and pandas names them: **split** the rows into groups,
**apply** a summary to each group, **combine** the results into a new table.

In [ ]:
clean.groupby("success").size()

That is `value_counts()` with extra steps, and in fact
`value_counts()` is the shortcut for exactly this.

In [ ]:
print(clean.success.value_counts())
print()
print(clean.success.value_counts(normalize=True).round(3))    # as proportions

The interesting version summarizes a **different** column than the
one you grouped by.

In [ ]:
clean.groupby("success")[["logins_per_week", "ug_gpa", "course_score"]].mean().round(2)

> **Predict.** Look at that table. Which of the three columns separates the two groups most sharply, relative to its own scale?
>
> Commit to an answer before you run the next cell.

**What happened.** `logins_per_week` more than triples between the groups. `ug_gpa`
moves too, but by a much smaller fraction of its range.

That is finding A from the capstone brief, visible in a single line of pandas:
**engagement separates outcomes better than prior grades do.** You are not
entitled to call it causal, and section 14 comes back to why. But it is the
first time in this course the claim has been something you can see rather than
something you were told.

## Several statistics at once

`.agg()` takes a list of functions, or a dictionary mapping columns to the
statistics you want from each.

In [ ]:
clean.groupby("success").logins_per_week.agg(["count", "mean", "median", "std"]).round(2)

In [ ]:
clean.groupby("success").agg({
    "logins_per_week": ["mean", "std"],
    "ug_gpa":          "mean",
    "course_score":    ["min", "max"],
}).round(2)

## Grouping by more than one column

Pass a list, and the result gets one index level per grouping column - a
**MultiIndex**.

In [ ]:
by_two = clean.groupby(["need_based_aid", "success"]).course_score.mean().round(1)
by_two

In [ ]:
print(type(by_two.index))
print()
print(by_two.unstack())        # pivot the inner level out into columns

**Takeaway.** **`.unstack()` turns the innermost index level into columns.**

A grouped result with two keys is hard to read as a tall list and easy to read
as a grid. `.unstack()` is how you get from one to the other, and
`.reset_index()` is how you get back to an ordinary flat table when something
downstream wants columns rather than an index.

## The bias features, seen for the first time

`need_based_aid` and `hs_pop_density` are in this extract for a reason, and the
capstone makes you decide what to do about them. Group by one and look.

In [ ]:
aid = clean.groupby("need_based_aid").agg(
    students=("student_id", "count"),
    mean_score=("course_score", "mean"),
    success_rate=("success", "mean"),
).round(3)

aid

**Takeaway.** **That gap is real, and it is not a measure of ability.**

`need_based_aid` is a socioeconomic flag. It correlates with the outcome because
disadvantage has real effects, which means a model handed this column will use
it and will get *more accurate* by doing so.

That is the trap, and it is worth naming now rather than at the capstone: a
feature can carry genuine signal and still be one you must refuse to use.
Excluding it costs accuracy. That cost is the decision, not an argument against
making it.

**Your turn.**

Note the named-argument form used above: `students=("student_id", "count")`.
That is **named aggregation**, and it produces readable column names instead of
a MultiIndex on the columns.

Use it to build a table grouped by `hs_sports` showing, for each group: the
number of students, mean `ug_gpa`, mean `logins_per_week`, and success rate.
Round to 3 decimals. Then say in a comment whether playing a high-school sport
looks like it matters.

In [ ]:
### ENTER CODE HERE ###

Everything so far used one flat file. That is not how the
data lives.

Institutional Research does not keep one table. The registrar owns high-school
records, the LMS owns engagement, the provost's office owns outcomes, and each
system exports its own file keyed on `student_id`. The flat extract you have
been cleaning was produced by joining them - which is where its two duplicate
rows came from.

This section does that join yourself, and then checks your answer against the
extract.

---
# 11 · Merging

Five files, one key. Load them and look at the shapes.

In [ ]:
students      = pd.read_csv("data/students.csv")
high_school   = pd.read_csv("data/high_school.csv")
undergraduate = pd.read_csv("data/undergraduate.csv")
lms_activity  = pd.read_csv("data/lms_activity.csv")
outcomes      = pd.read_csv("data/outcomes.csv")

for name, t in [("students", students), ("high_school", high_school),
                ("undergraduate", undergraduate), ("lms_activity", lms_activity),
                ("outcomes", outcomes)]:
    print(f"{name:<15} {t.shape[0]:>4} rows x {t.shape[1]} cols   {list(t.columns)}")

## The key has to be checked before you use it

A merge on a non-unique key multiplies rows. Check first, every time.

In [ ]:
for name, t in [("students", students), ("high_school", high_school),
                ("undergraduate", undergraduate), ("lms_activity", lms_activity),
                ("outcomes", outcomes)]:
    print(f"{name:<15} student_id unique: {t.student_id.is_unique}")

## One merge

In [ ]:
pair = students.merge(high_school, on="student_id")

print(pair.shape)
pair.head(3)

## The four kinds of join

`how=` decides what happens to rows that have no match on the other side.

In [ ]:
left  = pd.DataFrame({"student_id": [1, 2, 3], "name": ["A", "B", "C"]})
right = pd.DataFrame({"student_id": [2, 3, 4], "score": [90, 80, 70]})

for how in ["inner", "left", "right", "outer"]:
    out = left.merge(right, on="student_id", how=how)
    print(f"{how:<7} {len(out)} rows   ids {sorted(out.student_id)}")

> **Predict.** The default is `how="inner"`. If one of the five real tables were missing a student, what would an inner join silently do to them?
>
> Commit to an answer before you run the next cell.

**What happened.** Drop them, with no error and no warning.

That is the merge equivalent of `dropna()` from section 8, and it is dangerous
for the same reason: the students most likely to be missing from a system are
rarely a random sample of the cohort. A student with no LMS record could vanish
from your analysis entirely, and the row count is the only thing that would tell
you.

**Check the row count after every merge.** If it went down, find out who left.
If it went up, your key was not unique.

**Takeaway.** **`how="left"` is the safer default when one table is the
spine.** It keeps every row of the left table and fills gaps with `NaN`, so
missing matches become visible missing data rather than absent rows.

Use `how="inner"` when you genuinely require both sides, and say so.

## Merging all five

In [ ]:
from functools import reduce

tables = [students, high_school, undergraduate, lms_activity, outcomes]
rebuilt = reduce(lambda a, b: a.merge(b, on="student_id", how="left"), tables)

print(f"shape: {rebuilt.shape}")
print(f"columns: {list(rebuilt.columns)}")

## Checking the answer

`clean` came from the flat extract, cleaned. `rebuilt` came from joining the
five source tables. They should describe the same 100 students.

We can compare them directly, which is a habit worth forming: when two routes
should give the same answer, make them prove it.

In [ ]:
a = rebuilt.sort_values("student_id").reset_index(drop=True)
b = clean.sort_values("student_id").reset_index(drop=True)

print(f"same shape:   {a.shape == b[a.columns].shape}")
print(f"same ids:     {a.student_id.equals(b.student_id)}")
print()
for col in ["hs_gpa", "ug_gpa", "logins_per_week", "course_score"]:
    print(f"{col:<18} identical: {a[col].equals(b[col])}")

> **Predict.** `submission_rate` is not in that list. Predict whether it would match, and why.
>
> Commit to an answer before you run the next cell.

In [ ]:
print("rebuilt: ", rebuilt.submission_rate.dtype, rebuilt.submission_rate.dropna().head(2).tolist())
print("clean:   ", clean.submission_rate.dtype, clean.submission_rate.dropna().head(2).tolist())

**What happened.** They hold the same information and they are not equal.

`rebuilt` came straight from `lms_activity.csv`, so `submission_rate` is still
the raw `"84%"` string. `clean` went through section 7, so it is a float.

Nothing is wrong. It is a reminder that **merging is not cleaning** - joining
five raw tables gives you one raw table, and every cleaning decision from
sections 6 to 8 still has to be applied afterwards. In a real pipeline the
cleaning is written once as a function and called on whatever arrives.

**Your turn.**

The flat extract has 102 rows; the merged tables have 100. Section 5 already
told you why.

1. Merge `students` and `outcomes`, keeping every student.
2. Confirm the result has 100 rows.
3. In a comment, explain where the extra two rows in `cohort_extract.csv` came
   from, and why the source tables do not have them.

In [ ]:
### ENTER CODE HERE ###

---
# 12 · Pivot tables

A pivot table is a `groupby` arranged as a grid: one variable down the side,
another across the top, a statistic in the cells. Everything here could be done
with `groupby` and `.unstack()`; `pivot_table` is the readable shorthand.

In [ ]:
clean.pivot_table(
    index="need_based_aid",
    columns="hs_sports",
    values="course_score",
    aggfunc="mean",
).round(1)

Read it as a sentence: mean `course_score`, for each combination of
`need_based_aid` down the side and `hs_sports` across the top.

Add margins for the row and column totals.

In [ ]:
clean.pivot_table(
    index="need_based_aid",
    columns="success",
    values="course_score",
    aggfunc=["mean", "count"],
    margins=True,
).round(1)

**Takeaway.** **Always show the counts.** A pivot of means with no `count`
invites you to compare two cells that might be built from forty students and
three.

`aggfunc=["mean", "count"]` costs one word and makes the table honest.

## Binning a continuous variable

Pivot tables want categories. `pd.cut` turns a numeric column into bands.

In [ ]:
clean = clean.copy()
clean["engagement_band"] = pd.cut(
    clean.logins_per_week,
    bins=[-0.01, 1.0, 2.5, 4.0, 10.0],
    labels=["very low", "low", "moderate", "high"],
)

clean.engagement_band.value_counts().sort_index()

> **Predict.** Some students have no `logins_per_week` at all. Which band do you think they landed in?
>
> Commit to an answer before you run the next cell.

In [ ]:
print(f"missing logins:      {clean.logins_per_week.isna().sum()}")
print(f"missing band:        {clean.engagement_band.isna().sum()}")
print(f"rows counted above:  {clean.engagement_band.value_counts().sum()} of {len(clean)}")

**What happened.** None of them. `pd.cut` returns `NaN` for a value it cannot
place, so the missing students are missing from the bands too.

This is the correct behavior and it is easy to miss, because
`value_counts()` **excludes NaN by default** - the bands do not add up to the
number of students, and nothing on screen says so unless you check. Read the
three numbers the cell just printed rather than taking my word for it.

`value_counts(dropna=False)` shows them. Get in the habit: any time a count
matters, confirm it sums to the number of rows you started with.

In [ ]:
clean.engagement_band.value_counts(dropna=False).sort_index()

Now the pivot is worth reading.

In [ ]:
clean.pivot_table(
    index="engagement_band",
    values=["course_score", "success"],
    aggfunc=["mean", "count"],
    observed=True,
).round(3)

**Your turn.**

Build a pivot table showing mean `course_score` with `engagement_band` down the
side and `need_based_aid` across the top, including counts.

Then say in a comment which cell you trust least, and why.

In [ ]:
### ENTER CODE HERE ###

---
# 13 · Errors you will actually hit

Module 2 taught you to read a traceback from the bottom up. pandas errors follow
that rule, but the vocabulary is new and a few of the most common ones are not
errors at all - they are silently wrong answers.

Start with the loud ones.

### `KeyError` - no such label

In [ ]:
%%expect KeyError
clean["Course_Score"]

Column names are case-sensitive and whitespace-sensitive.
`clean.columns.tolist()` settles it in one line, and `df.columns.str.strip()`
fixes a file whose headers arrived with stray spaces.

### `ValueError` - ambiguous truth value

In [ ]:
%%expect ValueError
if clean.ug_gpa > 3.0:
    print("yes")

Section 9. A Series of a hundred booleans is not one condition.
You want `.any()`, `.all()`, or a mask.

### `TypeError` - arithmetic on text

In [ ]:
%%expect TypeError
raw.submission_rate.sum() / len(raw)

Section 7. Check `.dtype` before you compute.

## The quiet ones

These do not raise. They just give you the wrong number.

### A count that silently excludes missing values

In [ ]:
print(f"rows in the table:              {len(clean)}")
print(f"engagement_band.value_counts(): {clean.engagement_band.value_counts().sum()}")
print(f"mean of a column with NaN:      {clean.logins_per_week.mean():.3f}")
print(f"  computed over:                {clean.logins_per_week.notna().sum()} rows, not {len(clean)}")

**Takeaway.** **Almost every pandas statistic skips missing values without
telling you.** `.mean()`, `.sum()`, `.count()`, `value_counts()` - all of them.

That is usually what you want, and it is never what you should *assume*. The
number of rows a statistic was computed over is part of the statistic. Report
it, or at least look at it.

### A merge that changed the row count

In [ ]:
dupe_key = pd.DataFrame({"student_id": [1003, 1003], "note": ["x", "y"]})
before = len(clean)
after = len(clean.merge(dupe_key, on="student_id", how="left"))

print(f"rows before: {before}")
print(f"rows after:  {after}")

**What happened.** One student matched two rows on the right, so pandas produced
a row for each combination. No error - a merge is *supposed* to do this, and
sometimes you want it.

At this scale you would notice. On a real join, a key that repeats a few times
in a large table inflates the row count by an amount that looks plausible, and
every average computed afterwards is quietly weighted toward the duplicated
students.

### Chained assignment

Section 6 covered this one. The tell is `][` on the left of an `=`, the symptom
is an edit that does nothing, and the fix is a single `.loc[rows, cols] =`.

## A diagnostic habit

Four questions that catch most of it, and none of them take longer than a
second:

In [ ]:
def sanity(df, label=""):
    """Print the four things worth checking after any transformation."""
    print(f"--- {label}")
    print(f"  shape        {df.shape}")
    print(f"  duplicated   {df.duplicated().sum()}")
    print(f"  any missing  {df.isna().any().any()}")
    bad = [c for c in df.columns if df[c].dtype == object]
    print(f"  object cols  {bad}")


sanity(raw, "raw extract")
sanity(clean, "after cleaning")

**Takeaway.** **Run something like `sanity()` after every step that changes
the shape of your data.**

It is four lines and it turns three of the failures in this section into
something you notice immediately rather than three steps later, when the cause
is no longer on screen.

What follows is the module in one piece. Each challenge is
a question Lee could plausibly ask, and answering it takes the whole
chain - load, diagnose, clean, filter, group, report.

Nothing here needs a method you have not met. If you find yourself wanting one,
the problem is probably the question rather than the tooling.

---
# 14 · Challenges

Work from the raw extract each time, so the cleaning is part of the answer.

**Challenge 1 - the engagement report.**

Produce one table for Lee with a row per engagement band and these columns:
number of students, mean course score, success rate, and the share of the cohort
in that band. Include the students with no LMS record as their own row rather
than dropping them.

In [ ]:
### ENTER CODE HERE ###

**Challenge 2 - does prior achievement or engagement predict better?**

Compute the correlation between `course_score` and each of `ug_gpa`,
`hs_gpa`, and `logins_per_week`. Report them in one small table, sorted, and say
in a comment what you are *not* entitled to conclude.

In [ ]:
### ENTER CODE HERE ###

**Challenge 3 - the audit trail.**

A colleague will re-run your work. Write a single function
`load_and_clean(path)` that goes from the raw CSV to the cleaned DataFrame,
applying every decision from sections 6 to 8, and returns it. Then prove it
reproduces `clean`.

In [ ]:
### ENTER CODE HERE ###

That last point is the module, really.

Loading and cleaning look like preparation - the dull part before the analysis
starts. They are not. Every choice you made in sections 6 to 8 changed a number
somebody will eventually quote, and the only thing separating an analysis from
an assertion is whether you can say what those choices were.

Module 5 turns these tables into charts. The capstone hands you the same extract
and asks for findings, and it will hold you to the decisions you recorded
here.